# Lesson 2.1 — From Perception Output to World State

Perception emits **raw detections**; the Understand stage converts them into a **trusted world state** by de-duplicating, annotating, and normalizing. We verify the two guarantees: *no duplicate passes, no real fruit is lost*.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception,
    dedupe, understand)
checks = []
world = GreenhouseWorld.demo_row(n=5, seed=4)
# make every fruit ripe & reachable so the only effect we study is de-duplication
from modules.module09.integration import REACH_MIN, REACH_MAX
for i, f in enumerate(world.fruit):
    f.ripe = True
    r = 0.6 + 0.15*i           # place safely inside the reach annulus
    f.x, f.y = r, 0.3
real_ids = sorted(f.fid for f in world.fruit)
print('real fruit:', real_ids)


real fruit: ['F0', 'F1', 'F2', 'F3', 'F4']


In [2]:
# perceive WITH a duplicate of F2 and a little noise
rng = np.random.default_rng(4)
dets = model_perception(world, rng=rng, noise=0.01, duplicate=['F2'])
print('raw detection count:', len(dets), '(includes a duplicate of F2)')
checks.append(len(dets) == len(world.fruit) + 1)   # one extra = the duplicate


raw detection count: 6 (includes a duplicate of F2)


### The conversion: dedupe + annotate
`understand()` builds the world state. After conversion, the duplicate is gone and every *real* fruit survives, each annotated with reachability and distance.

In [3]:
targets, current = understand(dets, world)
got_ids = sorted(set(t['id'] for t in targets))
print('world-state ids:', got_ids)
print('example annotation:', {k: (round(v,3) if isinstance(v,float) else v)
      for k,v in targets[0].items() if k in ('id','reachable','dist','ripe')})
checks.append(got_ids == real_ids)                 # no real fruit lost
checks.append(len(targets) == len(world.fruit))    # duplicate collapsed
checks.append(all('reachable' in t and 'dist' in t for t in targets))  # annotated


world-state ids: ['F0', 'F1', 'F2', 'F3', 'F4']
example annotation: {'id': 'F0', 'ripe': True, 'reachable': True, 'dist': 0.18}


In [4]:
# direct dedupe check: two near-identical points collapse to one
a = {'id':'X','xy':np.array([1.00,0.50]),'ripe':True,'confidence':0.9}
b = {'id':'X','xy':np.array([1.03,0.49]),'ripe':True,'confidence':0.8}
far = {'id':'Y','xy':np.array([0.10,0.60]),'ripe':True,'confidence':0.9}
merged = dedupe([a,b,far], tol=0.08)
print('dedupe([X,X,Y]) ->', [m['id'] for m in merged])
checks.append(len(merged) == 2)   # the two X collapse, Y stays
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


dedupe([X,X,Y]) -> ['X', 'Y']
5/5 checks passed.
All checks passed.
